# Exercise — Audit & Remediate Catalog Metadata (Customer & Marketing)

**Trailhead Provisions** is under a GDPR audit notice. Using the **AWS Glue Data Catalog**, find
where governance controls are missing for the two domains that hold personal data —
**Customer Identity & Loyalty** and **Marketing & Campaigns** — **fix the gaps in the catalog**,
verify with a re-audit, then write a short reconciliation memo. See `INSTRUCTIONS.md`.

## Step 0: Connect to your Cloud Lab
Run the cell below to configure your AWS Cloud Lab environment and seed the AWS Glue Data Catalog (`trailhead`) if it is not yet present in your lab account.

In [ ]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

# Auto-load .env from the workspace or parent directories if present
from pathlib import Path
for _parent in (Path.cwd(), *Path.cwd().parents):
    _env_file = _parent / ".env"
    if _env_file.is_file():
        for _raw in _env_file.read_text(encoding="utf-8").splitlines():
            _line = _raw.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                _k, _v = _k.strip(), _v.strip().strip("\"").strip("'")
                if _k and _v and not os.environ.get(_k):
                    os.environ[_k] = _v
        break

import boto3
from botocore.exceptions import ClientError, NoCredentialsError
import setup_aws

_CRED_MSG = (
    "AWS credentials are missing or expired.\n"
    "1. Open Cloud Resources and start your Cloud Lab.\n"
    "2. Copy AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, and AWS_SESSION_TOKEN.\n"
    "3. Uncomment and paste them into Step 0 above, then re-run Step 0 (and re-run in each new lab session)."
)

# Verify credentials before seeding
sts = boto3.client("sts", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    sts.get_caller_identity()
except NoCredentialsError:
    raise RuntimeError(_CRED_MSG) from None
except ClientError as exc:
    code = exc.response.get("Error", {}).get("Code", "")
    if code in {"ExpiredToken", "ExpiredTokenException", "InvalidClientTokenId", "UnrecognizedClientException", "AuthFailure", "AccessDeniedException"}:
        raise RuntimeError(_CRED_MSG) from None
    raise
# Reset/seed the Glue Data Catalog, S3 tables, and Lake Formation enforcement to a clean baseline
setup_aws.main()

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog()
gc.reset()
print(f"Backend: {lf_backend()} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")

## 1. Run the catalog audit (provided)

In [ ]:
audit = gc.catalog_audit()
print('Total gaps across all domains:', len(audit))
audit

## 2. Narrow to the Customer & Marketing domains
Filter the audit to the two domains you're responsible for, and inspect the column descriptions of `customer` and `campaign_membership` to see what each untagged column actually stores.

In [ ]:
# TODO(you): filter `audit` to the two domains you own and display the result.
my_domains = ["Customer Identity & Loyalty", "Marketing & Campaigns"]
my_gaps = None   # audit[audit["domain"].isin(...)]
my_gaps

In [ ]:
# Provided: inspect column descriptions in your domains to see what each column stores.
print("--- customer columns ---")
display(gc.describe("customer"))
print("--- campaign_membership columns ---")
display(gc.describe("campaign_membership"))

## 3. Remediate the gaps in the catalog
Write the missing governance metadata back to Glue with the provided helpers:
- `gc.set_table_metadata(table, owner=..., classification=..., retention=...)`
- `gc.tag_column(table, column, classification)`

Valid platform classification values are **`PII`**, **`SENSITIVE`**, and **`PUBLIC`**. Be sure to classify personal-data columns (and PII-bearing tables) as **`PII`**.

In [ ]:
# TODO(you): remediate EVERY gap in `my_gaps`, writing to the catalog.
#   gc.set_table_metadata(table, owner=..., classification=..., retention=...)
#   gc.tag_column(table, column, classification)   # valid classifications: "PII", "SENSITIVE", "PUBLIC"
print("TODO: write your remediation calls above")

## 4. Re-audit to verify (provided)
Confirm your two domains have 0 remaining gaps and that every personal-data column carries the `PII` classification.

In [ ]:
from governance_toolkit import PII_COLUMN_NAMES

re_audit = gc.catalog_audit()
remaining = re_audit[re_audit["domain"].isin(my_domains)]
print("Gaps in your domains -> before:", len(my_gaps) if my_gaps is not None else "?",
      "after:", len(remaining))
assert len(remaining) == 0, f"Expected 0 remaining gaps in {my_domains}, got {len(remaining)}"

untagged_pii = [
    f"{t}.{c.name}"
    for t in gc.tables()
    if gc.catalog.table(t).domain in my_domains
    for c in gc.catalog.table(t).columns
    if c.name in PII_COLUMN_NAMES and c.classification != "PII"
]
assert not untagged_pii, (
    f"Personal-data column(s) in {my_domains} must be tagged 'PII' (found {len(untagged_pii)} untagged)"
)
print("All personal-data columns in your domains are tagged 'PII': True")
remaining

## 5. Review the cross-domain conflicts (provided)

In [ ]:
gc.semantic_conflicts()

## 6. Write your reconciliation memo
Replace the cell below with your memo. Address every requirement in `INSTRUCTIONS.md`.

> **Your reconciliation memo here.** Address:
>
> - The untagged PII column(s) you found and fixed, and why each was the top GDPR risk.
> - The table-level gaps (missing owner / classification / retention) you remediated and what each meant.
> - The cross-domain semantic conflict(s) that involve these domains.
> - The tags/standards you recommend the platform enforce so the gaps don't recur.